### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/10_Collect_Thesis_Results.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Fnotebooks%2Fthesis%2F10_Collect_Thesis_Results.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Fnotebooks%2Fthesis%2F10_Collect_Thesis_Results.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/10_Collect_Thesis_Results.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.

Frozen training and collection notebooks also require the campaign artifacts prepared in notebook **01**.

See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.


In [ ]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME,
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "notebooks" / "init.py" for path in _locations
                     if (path / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, repository=REPOSITORY, revision=REVISION,
    runtime=RUNTIME, cuda=CUDA,
)


# 10 — Export the saved results chapter package

Run this after all **24 frozen benchmark streams** finish. It authenticates saved results, recovers valid completion bookkeeping when needed, and creates a compact ZIP for thesis writing. It does not train, load a checkpoint, generate images or compute new predictions.

Use a **TensorFlow 2.20 / Keras 3** kernel.

In [ ]:
from IPython.display import display

from notebooks.thesis.workflow import check_runtime


print(check_runtime())
import json
import pandas as pd
from IPython.display import FileLink
from notebooks.thesis.results_package import export_results_package

The final package requires three paired seeds `[1103, 2207, 3301]`, full five/ten-task schedules and every declared condition. Keep the development seed 17 and older two-seed campaigns separate. `PROGRESS = True` creates an explicitly marked progress package without native final inference; use it only to inspect already completed streams. If a progress snapshot already exists with fewer completed runs, choose a new `OUTPUT` directory to preserve it.

In [ ]:
CAMPAIGN = ROOT / "results/thesis_route_one/minimum_v5_tf220"
PROGRESS = False
DETAILS = False  # True adds optional diagnostic tables and figures; choose a new OUTPUT.
OUTPUT = None  # Default: writing_package (final) or progress_package.
package = export_results_package(CAMPAIGN / "frozen_design.json",
                                 progress=PROGRESS, output_dir=OUTPUT, details=DETAILS)
print("Package:", package["directory"])
print("ZIP:", package["zip"])
display(FileLink(str(package["zip"])))

Inspect the main results and primary paired effects. All comparable outcomes are computed within each stream first, then summarized with mean, sample SD (`ddof=1`) and actual n. Blank observations remain unavailable. The native primary 95% paired interval is retained separately from SD.

The compact table includes local backward transfer (final minus acquisition accuracy), which differs from TMCL's reference-model transfer metric.

In [ ]:
summary = json.loads((package["directory"] / "RESULT_SUMMARY.json").read_text("utf-8"))
print(summary["status"], "—", summary["completed_streams"], "completed streams")
for name in ("thesis_summary",):
    print(name.replace("_", " "))
    display(pd.DataFrame(summary["tables"][name]))
if summary["native_primary_statistics"]:
    display(pd.read_csv(package["directory"] / "tables/T90_primary_native_interval.csv"))
else:
    print("Progress view: no final paired confidence interval yet.")

Read the compact treatment summary and the primary learned-minus-extra-joint paired interval first. The default ZIP retains full numeric source rows, configuration, run identities and hashes. Set `DETAILS=True` with a new output directory only when the additional validation diagnostics, trajectories and saved replay figures are useful for writing.

Every complete stream is an independent replicate; three pairs give limited precision. Test outcomes describe efficacy, validation measurements describe mechanisms, and replay images are qualitative. Preserve negative and unavailable outcomes. Collection never trains or changes original results. Write from the saved observations and retain these limits.